# Async distillation: a sanity check on one example

**Qwen3-4B solves 17 × 23 → check the answer → train Qwen3-0.6B → run the student independently.**

Run the six cells from top to bottom using the project environment.
`demo.py` orchestrates collection and training; `rollout.py` contains the async schedule and checks.
Generation uses the original `async_thoughts`; training uses the existing replay in `minisgl`.
Full logs and checkpoints are saved under `RUN`.


## 1. Settings

Select one GPU before starting the notebook. Device, thread, and model-cache settings are inherited.
Teacher and student use the same token budgets and probe period. Each execution
of this cell creates a new directory and preserves previous results.


In [ ]:
import json
import tempfile
from pathlib import Path

ROOT = Path.cwd().resolve()
from transformers import AutoTokenizer

from minisgl_examples.training.distillation import demo
from minisgl_examples.training.process import isolated
from minisgl_examples.training.student import save_example

TEACHER = "Qwen/Qwen3-4B"
STUDENT = "Qwen/Qwen3-0.6B"
PROMPT = r"What is 17 * 23? Give one short calculation and put only the final integer in \boxed{...}."
EXPECTED = [17 * 23]  # independent reference: 391
LIMITS = {"think_tokens": 128, "answer_tokens": 192, "probe_period": 4,
          "memory_ratio": 0.7, "max_seq_len": 8192, "cuda_graphs": True}
STEPS, LR = 100, 1e-6

runs = ROOT / "training_runs/async_distillation_sanity_check"
runs.mkdir(parents=True, exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix="simple_multiply_", dir=runs))
tokenizer = AutoTokenizer.from_pretrained(TEACHER)
print("Run directory:", RUN)


## 2. Generate a correct teacher rollout

The 4B model generates thoughts and an answer through the original `async_thoughts` in BF16.
The trace records all requests, cache contexts, and sampled tokens. Independently check
the final boxed answer: it must be 391. An incorrect or truncated answer stops
execution **before student training**.


In [ ]:
teacher = isolated(
    demo.collect_trace, TEACHER, prompt=PROMPT,
    log_path=RUN / "teacher/generation.log", **LIMITS,
)
teacher_trace = save_example(teacher, RUN / "teacher/trace.json")
print(tokenizer.decode(teacher["writer_tokens"], skip_special_tokens=True))

teacher_grade = demo.check_answer(teacher, tokenizer, EXPECTED)
print("Teacher check:", teacher_grade["status"])
assert teacher_grade["status"] == "PASS", teacher_grade
print("Supervised targets:", sum(t != -100 for t in teacher["targets"]))


## 3. Memorize one trajectory

Each optimizer step uses all supervised tokens in this example.
The student receives teacher tokens and the recorded context visibility (teacher forcing).
Loss is mean CrossEntropy over thinker/writer targets; prompts and probe requests have no direct loss.

Forward uses BF16 with sensitive operations in FP32. Master weights, their gradients,
and AdamW states remain FP32. Gradient checkpointing is enabled. The learning rate is constant;
evaluation runs every 5 updates. Overfit success: **accuracy = 100%, loss < 0.01**.


In [ ]:
metrics = isolated(
    demo.train_student, STUDENT, teacher, RUN / "trained",
    steps=STEPS, learning_rate=LR, log_path=RUN / "training.log",
)
print("Loss:", metrics["initial"]["loss"], "→", metrics["final"]["loss"])
print("Accuracy:", metrics["initial"]["accuracy"], "→", metrics["final"]["accuracy"])
print("Updates:", metrics["completed_steps"])
assert metrics["overfit_achieved"], "Overfit not reached; see training.log and metrics.json"
checkpoint = Path(metrics["checkpoint"])


## 4. Verify the saved weights

A fresh process loads the checkpoint from disk and repeats replay evaluation without updating weights.
This confirms that the checkpoint retains overfit. This is still teacher forcing, not free generation.


In [ ]:
reloaded = isolated(
    demo.verify_checkpoint, teacher_trace, checkpoint, log_path=RUN / "reloaded.log",
)
(RUN / "reloaded.json").write_text(json.dumps(reloaded))
assert reloaded["accuracy"] == 1.0 and reloaded["loss"] < 0.01, reloaded
assert abs(reloaded["loss"] - metrics["final"]["loss"]) < 1e-5
print("Saved checkpoint retains overfit:", reloaded["loss"], reloaded["accuracy"])


## 5. Let the student solve the task independently

Load the trained checkpoint in BF16. Pass the original prompt and the same token budgets.
The student generates its own tokens and a new schedule; the teacher trace is not passed to inference.
The orchestrator starts the thinker, and a probe using the same model permits or pauses the writer.
Probe decisions were not directly supervised during training.


In [ ]:
student = isolated(
    demo.collect_trace, str(checkpoint), prompt=PROMPT,
    log_path=RUN / "student/generation.log", **LIMITS,
)
student_trace = save_example(student, RUN / "student/trace.json")
print("THINKER:\n", tokenizer.decode(student["thinker_tokens"], skip_special_tokens=True))
print("WRITER:\n", tokenizer.decode(student["writer_tokens"], skip_special_tokens=True))


## 6. Three independent checks

- **Answer:** the boxed integer must be 391; the teacher is graded the same way.
- **Structural async behavior:** both streams run, the writer starts before the last
  thinker samples, and it reads a growing thinker context; the cache graph is valid and probe requests exist.
  This does not establish the optimality of individual `yes/no` decisions.
- **Exact rollout match:** tokens, targets, and cache history match after consistently
  renaming block IDs. Different schedules count as a mismatch even if the answers are identical.

Display every status, including `FAIL`. Full details are saved in `checks.json`.
Answer correctness, async behavior, and exact teacher reproduction are separate outcomes;
successful replay overfit does not guarantee successful independent generation.


In [ ]:
report = demo.compare_rollouts(teacher_trace, student_trace, tokenizer, EXPECTED, RUN / "checks.json")
labels = {
    "teacher_answer": "Teacher answer correctness",
    "student_answer": "1. Student answer correctness",
    "teacher_async": "Teacher async behavior",
    "student_async": "2. Student async behavior (structure)",
    "exact_recorded_rollout": "3. Exact rollout match",
}
for name, label in labels.items():
    print(f"{label}: {report[name]['status']}")
    if report[name]["status"] != "PASS":
        print("  " + report[name]["reason"])
        details = report[name]["details"]
        if "expected" in details:
            print("  Answer / reference:", details.get("actual"), "/", details["expected"])
        if details.get("failed"):
            print("  Failed checks:", ", ".join(details["failed"]))

for role, comparison in report["exact_recorded_rollout"]["details"]["streams"].items():
    print(f"{role}: equal={comparison['equal']}, common prefix={comparison['common_prefix_tokens']}, "
          f"tokens teacher/student={comparison['teacher_length']}/{comparison['student_length']}")
difference = report["exact_recorded_rollout"]["details"]["first_trace_difference"]
if difference:
    print("Trace mismatch:", json.dumps(difference, ensure_ascii=False))
print("Full report:", RUN / "checks.json")
